In [14]:
import numpy as np
import pickle

def reformat_locations_from_dict(data: dict) -> dict:
    """
    Reformats a nested dict (as shown in the screenshot) into the structured dictionary format
    matching reformat_locations_from_df output.
    """
    reformatted_data = {}

    for purpose, entries in data.items():
        if purpose not in reformatted_data:
            reformatted_data[purpose] = {
                'identifiers': [],
                'names': [],
                'coordinates': [],
                'potentials': []
            }

        for identifier, entry in entries.items():
            coords = entry.get('coordinates', [0.0, 0.0])
            name = entry.get('name', "")
            potential = entry.get('capacity', 0.0)  # Assuming 'capacity' is used as potential

            # Append to the lists
            reformatted_data[purpose]['identifiers'].append(identifier)
            reformatted_data[purpose]['names'].append(name)
            reformatted_data[purpose]['coordinates'].append(np.array(coords))
            reformatted_data[purpose]['potentials'].append(potential)

    # Convert lists to numpy arrays for consistency
    for purpose, d in reformatted_data.items():
        d['identifiers'] = np.array(d['identifiers'], dtype=object)
        d['names'] = np.array(d['names'], dtype=str)
        d['coordinates'] = np.array(d['coordinates'], dtype=float)
        d['potentials'] = np.array(d['potentials'], dtype=float)

    return reformatted_data


# Load input data
with open(r'C:\Users\petre\Documents\GitHub\MATSimPipeline\data\locations_data.pkl', 'rb') as f:
    input_data = pickle.load(f)

# Reformat data
output_data = reformat_locations_from_dict(input_data)
import folium
import numpy as np
import pyproj

def plot_locations_folium_epsg25832(locations_dict):
    # Setup transformer from EPSG:25832 to EPSG:4326
    transformer = pyproj.Transformer.from_crs(25832, 4326, always_xy=True)

    coords = []
    for data in locations_dict.values():
        coords.extend(data['coordinates'])

    # Convert all coordinates
    coords = np.array(coords)
    lonlat_coords = np.array([transformer.transform(x, y) for x, y in coords])

    mean_lat = lonlat_coords[:, 1].mean()
    mean_lon = lonlat_coords[:, 0].mean()

    # Create folium map
    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=12)

    for purpose, data in locations_dict.items():
        for x, y in data['coordinates']:
            lon, lat = transformer.transform(x, y)
            folium.CircleMarker(
                location=[lat, lon],
                radius=4,
                color='blue',
                fill=True,
                fill_color='blue',
                popup=folium.Popup(f"{purpose}", parse_html=True)
            ).add_to(m)

    # Fit map to bounds
    sw = [lonlat_coords[:, 1].min(), lonlat_coords[:, 0].min()]
    ne = [lonlat_coords[:, 1].max(), lonlat_coords[:, 0].max()]
    m.fit_bounds([sw, ne])

    return m

# Usage
m = plot_locations_folium_epsg25832(output_data)
m.save("locations_map.html")


# # Save reformatted data
# with open(r'C:\Users\petre\Documents\GitHub\MATSimPipeline\locations_data_reformatted.pkl', 'wb') as f:
#     pickle.dump(output_data, f)


In [15]:
import pandas as pd
import re

# === Step 1: Read the CSV ===
file_path = r"C:\Users\petre\Documents\GitHub\MATSimPipeline\data\intermediates\mobile_population_remove_unfeasible.csv"  # Update with your actual path
df = pd.read_csv(file_path)

# === Step 2: Parse coordinates with regex ===
def parse_coords(val):
    if pd.isna(val) or val.strip() == '':
        return [None, None]
    try:
        # Extract numbers using regex
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", val)
        if len(numbers) >= 2:
            return [float(numbers[0]), float(numbers[1])]
        else:
            print(f"Not enough numbers found in '{val}'")
            return [None, None]
    except Exception as e:
        print(f"Failed to parse '{val}': {e}")
        return [None, None]

# Apply to each location column
df[['from_x', 'from_y']] = df['from_location'].apply(parse_coords).to_list()
df[['to_x', 'to_y']] = df['to_location'].apply(parse_coords).to_list()
df[['home_x', 'home_y']] = df['home_location'].apply(parse_coords).to_list()

# === Step 3: Drop original location columns (optional) ===
df = df.drop(columns=['from_location', 'to_location', 'home_location'])

# === Step 4: Save the reformatted CSV ===
output_path = "../../data/intermediates/mobile_population_remove_unfeasible_reformatted.csv"  # Update as needed
df.to_csv(output_path, index=False)

print(f"Reformatted CSV saved to: {output_path}")
print(df.head())


Reformatted CSV saved to: ../../data/intermediates/mobile_population_remove_unfeasible_reformatted.csv
       H_ID     H_GEW      H_HOCH  MODE  BASISAUF  TEILSTP  M_CAR  H_ART  \
0  10000370  0.090438   23.683802     2         2        1      0      2   
1  10000370  0.090438   23.683802     2         2        1      0      2   
2  10000370  0.090438   23.683802     2         2        1      0      2   
3  10000370  0.090438   23.683802     2         2        1      0      2   
4  10003390  1.606772  420.778622     2         2        1      0      2   

   H_GR  hhgr_gr  ...  home_to_main_seconds  home_to_main_time_is_estimated  \
0     2        2  ...                1050.0                             1.0   
1     2        2  ...                1050.0                             1.0   
2     2        2  ...                1050.0                             1.0   
3     2        2  ...                1050.0                             1.0   
4     2        2  ...                 900.0  